# RNAseq data cleanup

loading the needed libraries and the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from pathlib import Path
mmc2_path = Path('data') / 'mmc2.csv'
rna_seq_data = pd.read_csv(mmc2_path)
rna_seq_data.head()

## an explorativ overview of the data

In [ ]:
print(rna_seq_data.describe())
print(rna_seq_data.isnull().sum())
print(rna_seq_data.nunique())

In [ ]:
rna_seq_data.info()

some additional libraries for plotting

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

### Creating a histogram of one cell type as an example

In [ ]:
sns.histplot(rna_seq_data["Tgd.g2+d17.LN"], bins=30, kde=True)

Because the histogram does not show any usefull information in the normal scale, I will perform a log-transformation of all RNAseq values and create a new data frame with them

In [ ]:
numeric_cols = rna_seq_data.select_dtypes(include=[np.number]).columns
print(numeric_cols)

In [ ]:
log_rna_seq = rna_seq_data.copy()
log_rna_seq[numeric_cols] = np.log1p(log_rna_seq[numeric_cols])
log_rna_seq.head()

In [ ]:
sns.histplot(log_rna_seq["Tgd.g2+d17.LN"], bins=30, kde=True)

After the log transformation there is more usefull information visible in the histogram

In [ ]:
for  x in numeric_cols:
    sns.histplot(log_rna_seq[x], bins=30, kde=True)

### Creating new data frames with only those cell types relevant to us

In [ ]:
Tgd_cells = ["Unnamed: 0" ,
            "MPP4.135+.BM" ,
            "preT.DN1.Th" ,
            "preT.DN2a.Th" ,
            "preT.DN2b.Th" ,
            "preT.DN3.Th" ,
            "T.DN4.Th" ,
            "Tgd.g1.1+d1.24a+.Th" ,
            "Tgd.g2+d1.24a+.Th" ,
            "Tgd.g2+d17.24a+.Th" ,
            "Tgd.Sp" ,
            "Tgd.g1.1+d1.LN" ,
            "Tgd.g2+d1.LN" ,
            "Tgd.g2+d17.LN"
            ]

In [ ]:
Tgd_rna = rna_seq_data[Tgd_cells].copy()
Tgd_rna.head()
log_Tgd_rna = log_rna_seq[Tgd_cells].copy()
log_Tgd_rna.head()

RNAseq data only containing cell types relevant to us

## Filtering out low expression / noise

Low values <1.5 are not due to a detectable expression level, but are rather caused by noise. Thereby they can be disregarded.

In [ ]:
noise_frac = (rna_seq_data[numeric_cols] < 1.5).sum() / rna_seq_data.shape[0]
print(noise_frac)

How much of the RNAseq data is noise?

In [ ]:
print(noise_frac.mean())

plotting the noise fraction for each cell type

In [ ]:
noise_frac_df = noise_frac.reset_index()
noise_frac_df.columns = ['cell_type', 'noise_fraction']

plt.figure(figsize=(15, 10))
sns.pointplot(data=noise_frac_df, x='cell_type', y='noise_fraction')
plt.xticks(rotation=90)
plt.ylabel('Fraction of values < 1.5')
plt.xlabel('Cell type')
plt.title('Noise fraction per cell type')

27.9% of all values are under the threshold of 1.5 and 31.1% under a threshold of 2. 
The fraction of low expressed Genes is similar over all cell types.

How many genes are expressed in more than 10% of all cell types?

In [ ]:
(((rna_seq_data.iloc[:, 1:] < 1.5).sum(axis=1) / 86) < 0.1).sum()


### filtering out genes which are only expressed in less than 25 % of cell types

In [ ]:
gene_expr_frac = (rna_seq_data.iloc[:, 1:] > 1.5).sum(axis=1) / (rna_seq_data.shape[1] - 1)
rna_seq_filtered = rna_seq_data[rna_seq_data.index.isin(gene_expr_frac[gene_expr_frac >= 0.25].index)].copy()
rna_seq_filtered.info()

In [ ]:
rna_seq_rare_genes = rna_seq_data[rna_seq_data.index.isin(gene_expr_frac[gene_expr_frac < 0.25].index)].copy()
rna_seq_rare_genes.info()

This data frame contains the genes, which were filtered out and are only expressed in a few cell types

## Creating a function for log-tranformation

In [ ]:
def log_transform(df):
    df_numeric_cols = df.select_dtypes(include=[np.number]).columns
    log_df = df.copy()
    log_df[df_numeric_cols] = np.log1p(log_df[df_numeric_cols])
    return log_df

### Using the log_transform function on the filtered values and creating a histogram

In [ ]:
log_rna_filtered = log_transform(rna_seq_filtered)
log_rna_filtered.head()

In [ ]:
sns.histplot(log_rna_filtered["Tgd.g2+d17.LN"], bins=30, kde=True)

## Combination of filtering rarely expressed genes and genes with a low max expression

In [ ]:
gene_max = rna_seq_data.iloc[:, 1:].max(axis=1)
print(gene_max)
mask = (gene_expr_frac >= 0.25) & (gene_max >= 10)
rna_filtered_2 = rna_seq_data[mask].copy()
rna_filtered_2.info()

Ceck if any rarely expressed genes have high expression in one cell type

In [ ]:
rna_seq_rare_genes[gene_max >= 10].info()

plotting the filtered data to see how its distribution changed

In [ ]:
log_rna_filtered_2 = log_transform(rna_filtered_2)
sns.histplot(log_rna_filtered_2["Tgd.g2+d17.LN"], bins=30, kde=True)

## Only an absolute filter

An absolute filter seemed to be the most useful, because genes being rarely expressed could be cell type specifc markers and should not be filtered out.

In [ ]:
rna_filtered_absolute = rna_seq_data[rna_seq_data.iloc[:, 1:].max(axis=1) >= 10].copy()
rna_filtered_absolute.info()
log_rna_filtered_absolute = log_transform(rna_filtered_absolute)
sns.histplot(log_rna_filtered_absolute["Tgd.g2+d17.LN"], bins=30, kde=True)

## Filtering by Variation

First I have checked how the variation is distributed.

In [ ]:
gene_var_coef = rna_seq_data.iloc[:, 1:].std(axis=1) / rna_seq_data.iloc[:, 1:].mean(axis=1)
sns.histplot(gene_var_coef, bins=30, kde=True)

filtering out genes with a low variation coefficiant (lowest 40%)

In [ ]:
var_coef_threshold = gene_var_coef.quantile(0.40)
rna_variable_genes = rna_seq_data[gene_var_coef >= var_coef_threshold].copy()

Creating a separate data frame only with genes, that have a high varaiation coefficiant (top 1000), because these could be useful to distinguish between cell types.

In [ ]:
top1000_threshold = gene_var_coef.nlargest(1000).index
rna_var_top1000 = rna_seq_data.loc[top1000_threshold].copy()

These scatter plots should visualize the effekt of the filtering and also reveal a potential releationship between mean expression and variation.

In [ ]:
plt.figure(figsize=(10,7))
plt.scatter(np.log2(rna_seq_data.iloc[:, 1:].mean(axis=1)), gene_var_coef, s=5, alpha=0.5)
plt.xlabel('log2(Mean Expression)')
plt.ylabel('Variance Coefficient')
plt.title('scatter plot of log2(Mean Expression) vs Variance Coefficient')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
plt.scatter(np.log2(rna_variable_genes.iloc[:, 1:].mean(axis=1)), rna_variable_genes.iloc[:, 1:].std(axis=1) / rna_variable_genes.iloc[:, 1:].mean(axis=1), s=5, alpha=0.5)
plt.xlabel('log2(Mean Expression)')
plt.ylabel('Variance Coefficient')
plt.title('variable genes: scatter plot of log2(Mean Expression) vs Variance Coefficient')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10,7))
plt.scatter(np.log2(rna_var_top1000.iloc[:, 1:].mean(axis=1)), rna_var_top1000.iloc[:, 1:].std(axis=1) / rna_var_top1000.iloc[:, 1:].mean(axis=1), s=5, alpha=0.5)
plt.xlabel('log2(Mean Expression)')
plt.ylabel('Variance Coefficient')
plt.title('top 1000 variable genes: scatter plot of log2(Mean Expression) vs Variance Coefficient')
plt.grid(True)
plt.show()

<div class="alert alert-block alert-info">
<b>Conclusion</b> The RNA-seq data provided is already filtered and usable. Additional filtering only leads to a loss of information. Therefore we did not apply any of filtering methods for further analysis.
</div>

## Coloring the scatter plot to visualize gene functions

The scatter plot from before shows a good distribution of genes. It would be interesting to see if genes with similar functions group together (e.g. housekeeping genes could have low variation and therefore should be at the bottom).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import gseapy as gp
from pathlib import Path
mmc2_path = Path('data') / 'mmc2.csv'
rna_seq_data = pd.read_csv(mmc2_path)
gene_var_coef = rna_seq_data.iloc[:, 1:].std(axis=1) / rna_seq_data.iloc[:, 1:].mean(axis=1)

I have first loaded the needed libraries and to determine the gene function I will be using the GO gene annotations

In [ ]:
go_genesets = gp.get_library(name='GO_Biological_Process_2023', organism='Mouse')

instead of every gene in one category (default in the GO gene annotations), it is better to have every category for each gene

In [ ]:
gene2terms = {}
for term, genes in go_genesets.items():
    for gene in genes:
        gene2terms.setdefault(gene, set()).add(term)


the categories are put into broader categories ("ribosom" can match "ribosome" and "ribosomal")

In [ ]:
category_keywords = {
    "housekeeping":  ["ribosom", "translation", "RNA processing",
                      "proteasome", "DNA repair", "cell cycle"],
    "immune":        ["immune", "T cell", "lymphocyte", "cytokine",
                      "interferon", "NF-kB", "antigen"],
    "metabolism":    ["metabolic", "biosynthesis", "oxidation",
                      "glycolysis", "lipid", "mitochondri"],
    "transcription": ["transcription", "chromatin", "histone", "epigenetic"],
    "signaling":     ["signaling", "kinase", "receptor", "MAPK", "PI3K"],
}

these steps are performed in the next cell:
- get GO terms
- loop over all GO terms of this gene
- check each keyword

In [ ]:
def assign_category(gene):
    terms = gene2terms.get(gene.upper(), set())   
    for category, keywords in category_keywords.items():
        for t in terms:                   
            for kw in keywords:           
                if kw.lower() in t.lower():
                    return category       
    return "other"                        

Creating a data frame with the columns: gene name, log2 mean expresssion, variance coef.  This data frame can be later used for plotting.<br>
I am everything log transformed and also not transformed.

In [ ]:
mean_expression = rna_seq_data.iloc[:, 1:].mean(axis=1)
meanlog2_expression = np.log2(rna_seq_data.iloc[:, 1:].mean(axis=1))
meanexp_varcoef = pd.DataFrame({"gene": rna_seq_data["Unnamed: 0"], "mean_expression": mean_expression, "gene_var_coef": gene_var_coef})
meanlog2exp_varcoef = pd.DataFrame({"gene": rna_seq_data["Unnamed: 0"], "mean_log2_expression": meanlog2_expression, "gene_var_coef": gene_var_coef})
meanlog2exp_varcoef.head()

In [ ]:
meanexp_varcoef["category"] = meanexp_varcoef["gene"].apply(assign_category)
meanexp_varcoef.head()
meanlog2exp_varcoef["category"] = meanlog2exp_varcoef["gene"].apply(assign_category)

I am using different colormaps to highlight each group of genes. These can then be quickly changed while creating the plot.

In [ ]:
color_map = {
    "housekeeping":  "#2196F370",  
    "immune":        "#F4433670",  
    "metabolism":    "#4CAF5070",  
    "transcription": "#FF980070",  
    "signaling":     "#9C27B070",  
    "other":         "#CCCCCC70",  
}

color_map_housekeeping = {  
    "housekeeping":  "#2196F3",  
    "immune":        "#F4433600",  
    "metabolism":    "#4CAF4F00",  
    "transcription": "#FF990000",  
    "signaling":     "#9B27B000",
    "other":         "#CCCCCC70",  
}

color_map_immune = {
    "housekeeping":  "#CCCCCC00",  
    "immune":        "#F44336",  
    "metabolism":    "#CCCCCC00",  
    "transcription": "#CCCCCC00",  
    "signaling":     "#CCCCCC00",  
    "other":         "#CCCCCC70",  
}

color_map_metabolism = {
    "housekeeping":  "#CCCCCC00",  
    "immune":        "#CCCCCC00",  
    "metabolism":    "#4CAF50",  
    "transcription": "#CCCCCC00",  
    "signaling":     "#CCCCCC00",  
    "other":         "#CCCCCC70",  
}

color_map_transcription = {
    "housekeeping":  "#CCCCCC00",  
    "immune":        "#CCCCCC00",  
    "metabolism":    "#CCCCCC00",  
    "transcription": "#FF9800",  
    "signaling":     "#CCCCCC00",  
    "other":         "#CCCCCC70",  
}

color_map_signaling = {
    "housekeeping":  "#CCCCCC00",  
    "immune":        "#CCCCCC00",  
    "metabolism":    "#CCCCCC00",  
    "transcription": "#CCCCCC00",  
    "signaling":     "#9C27B0",  
    "other":         "#CCCCCC70",  
}


#first plot: log tranformed expression values


fig, ax = plt.subplots(figsize=(10, 8))

for category in ["other"] + [k for k in color_map if k != "other"]:
    group = meanlog2exp_varcoef[meanlog2exp_varcoef["category"] == category]
    ax.scatter(
        group["mean_log2_expression"],
        group["gene_var_coef"],
        c=color_map_housekeeping[category],
        label=f"{category} (n={len(group)})",  
        s=5          
    )

ax.set_xlabel("log2(Mean Expression)")
ax.set_ylabel("Variance Coefficient")
ax.legend(markerscale=4, loc="upper right")
ax.set_title("Gene Categories by GO Annotation")
plt.tight_layout()
plt.show()

plotting without log transformation, but with log scale x-axis<br>
This should give roughly the same plot, but does not change the actual data as with log tranformation.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for category in ["other"] + [k for k in color_map if k != "other"]:
    group = meanexp_varcoef[meanexp_varcoef["category"] == category]
    ax.scatter(
        group["mean_expression"],
        group["gene_var_coef"],
        c=color_map_housekeeping[category],
        label=f"{category} (n={len(group)})",  
        s=5          
    )

ax.set_xlabel("Mean Expression")
ax.set_ylabel("Variance Coefficient")
ax.set_xscale("log")
ax.legend(markerscale=4, loc="upper right")
ax.set_title("Gene Categories by GO Annotation")
plt.tight_layout()
plt.show()

plotting with a cutoff at 5000 and 1000 and no log scale<br>
Should make the scatter plot more understandable, because genes with very high mean expression values are no longer included. This is equivalent to zooming in on the left part of the plot.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for category in ["other"] + [k for k in color_map if k != "other"]:
    group = meanexp_varcoef[meanexp_varcoef["category"] == category]
    ax.scatter(
        group["mean_expression"],
        group["gene_var_coef"],
        c=color_map_housekeeping[category],
        label=f"{category} (n={len(group)})",  
        s=5          
    )

ax.set_xlabel("Mean Expression")
ax.set_ylabel("Variance Coefficient")
ax.set_xlim(1, 5000)
ax.legend(markerscale=4, loc="upper right")
ax.set_title("Gene Categories by GO Annotation")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for category in ["other"] + [k for k in color_map if k != "other"]:
    group = meanexp_varcoef[meanexp_varcoef["category"] == category]
    ax.scatter(
        group["mean_expression"],
        group["gene_var_coef"],
        c=color_map[category],
        label=f"{category} (n={len(group)})",  
        s=5          
    )

ax.set_xlabel("Mean Expression")
ax.set_ylabel("Variance Coefficient")
ax.set_xlim(1, 1000)
ax.legend(markerscale=4, loc="upper right")
ax.set_title("Gene Categories by GO Annotation")
plt.savefig("tilmann_plots/gene_variance_over_mean_scatter(limit1000).png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()